# Etapa B · Experimentos complementarios del paper — Mac mini

Lo que le falta al manuscrito (`paper/manuscrito/main.tex`, partes en rojo):

| Paso | Qué | Para qué en el paper | Tiempo aprox. |
|---|---|---|---|
| 2 | **B3 · Tiempo de cómputo** (procedimiento de CEC2017: T0, T1, T2) | tabla de tiempos | ~20 min |
| 3 | **B2 · Búsqueda aleatoria** en las 71 funciones × 51 corridas × D10/D50 | línea base que pide Vermetten et al. (2024) | ~1 h |
| 4 | **B1 · Convergencia** de los 10 algoritmos en 8 funciones de CEC2017, 500k, 51 corridas | figura de convergencia; mide el gasto real de ABC | ~3–5 h |
| 5 | **B4 · Sensibilidad** de MAREA-2 a 6 parámetros en **CEC2013** (suite distinta), 50k, 15 corridas | separa diseño de evaluación; incluye x_mid y ν de Cobetia | ~2 h |

Copiar al mini (commit `227574d`): **`complementarias.py`** (nuevo). `marea2.py`, `experimento_config.py` y `competidores.py` deben ser los mismos del protocolo: la celda 0 lo comprueba con hashes.
El paso 4 compara el final de cada curva con los JSON del protocolo a 500k, así que esos JSON tienen que estar en `resultados_experimento/` del mini.

Todo se puede interrumpir y reanudar. Correr **el paso 2 con la máquina libre** (mide tiempos); los demás pueden ir de noche.
**Solo en el Mac mini.**

In [ ]:
import sys, os, hashlib, subprocess, opfunu

WORKERS   = 20
RUNS_CONV = 51      # corridas de convergencia (mismas semillas que el protocolo)
RUNS_SENS = 15
ESPERADO = {
    "complementarias.py":    "0ad77bab9f318998e34795df4d2b23ba346f999f21d37b9f646c9445113bab81",
    "marea2.py":             "10d7448e59082adf4f464068646d74d25d8b8f54f5317f9eb4dfed32dacfe64b",
    "experimento_config.py": "ef9c5af25d46afa681276efeb5d0e25c68290404dc6093e8d8783b3554a914d1",
    "competidores.py":       "783a7ada89ddf917332d91efcf8179a8c3c703797943eb415d14a7141c68487b",
}

def correr(*args):
    """Ejecuta un script con el mismo Python del kernel y muestra la salida en vivo."""
    p = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:
        if "pkg_resources" not in linea:
            print(linea, end="")
    return p.wait()

print("python :", sys.executable)
print("carpeta:", os.getcwd())
print("opfunu :", opfunu.__version__)
assert opfunu.__version__ == "1.0.4", "opfunu tiene que ser 1.0.4 (1.0.1 esta roto)"
for f, h in ESPERADO.items():
    real = hashlib.sha256(open(f, "rb").read()).hexdigest()
    print(f"{f:24s} {real[:16]}  {'OK' if real == h else 'DISTINTO'}")
    assert real == h, f"{f} no es la version del commit 227574d: copiala de nuevo"
n500 = sum(os.path.exists(f"resultados_experimento/{a}_D{d}_FES500000.json")
           for a in ["MAREA2_A2_hgt", "LSHADE", "JSO", "DE", "PSO", "ABC", "GWO", "WOA", "FA", "CLONALG"]
           for d in (10, 50))
print(f"\nJSON del protocolo a 500k presentes: {n500}/20")
print("Entorno y archivos OK")

## 1 · Chequeo rápido (segundos)
El contador de evaluaciones no cambia el resultado de ningún algoritmo; búsqueda aleatoria y sensibilidad corren.

In [ ]:
rc = correr("complementarias.py", "check")
assert rc == 0, "el chequeo fallo: no seguir, mandar la salida a Claude"
CHEQUEADO = True

## 2 · Tiempo de cómputo, B3 (~20 min) — **con la máquina libre**
Secuencial (sin workers). No correr otras celdas ni otros notebooks mientras tanto.

In [ ]:
assert globals().get("CHEQUEADO"), "primero la celda de chequeo"
rc = correr("complementarias.py", "tiempo")
assert rc == 0

## 3 · Búsqueda aleatoria, B2 (~1 h)
Escribe `resultados_experimento/RANDOM_D{10,50}_FES{5000,50000,500000}.json`.

In [ ]:
assert globals().get("CHEQUEADO"), "primero la celda de chequeo"
rc = correr("complementarias.py", "random", "--workers", str(WORKERS))
assert rc == 0

## 4 · Convergencia, B1 (~3–5 h; se puede dejar de noche)
8 funciones de CEC2017 (F1, F3, F5, F10, F13, F17, F21, F26), 10 algoritmos, D10 y D50, 500k.

In [ ]:
assert globals().get("CHEQUEADO"), "primero la celda de chequeo"
rc = correr("complementarias.py", "conv", "--workers", str(WORKERS), "--runs", str(RUNS_CONV))
assert rc == 0

## 5 · Sensibilidad en CEC2013, B4 (~2 h)
25 configuraciones (la de referencia + 6 parámetros × 4 valores), D10 y D50, 50k.

In [ ]:
assert globals().get("CHEQUEADO"), "primero la celda de chequeo"
rc = correr("complementarias.py", "sens", "--workers", str(WORKERS), "--runs", str(RUNS_SENS))
assert rc == 0

## 6 · Verificar y traer los resultados al MacBook
Tiene que terminar en **TODO OK**. Las líneas "evaluaciones gastadas" de ABC son el dato que falta en el manuscrito: mandarlas tal cual.

Traer al MacBook:
- la carpeta `resultados_experimento/complementarias/` completa,
- los 6 `resultados_experimento/RANDOM_*.json` (a `resultados_experimento/`, **no a la raíz**).

In [ ]:
rc = correr("complementarias.py", "verificar", "--runs", str(RUNS_CONV))
print("\nverificacion:", "OK" if rc == 0 else "REVISAR")